In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
%run /Workspace/DE_Project_Pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg_de_project", "Catalog")
dbutils.widgets.text("data_source", "products_non_aws_new", "Data Source")

In [0]:
catalog=dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")

In [0]:
catalog=dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")

print(catalog, data_source)

# uncomment this when u set up AWS
# base_path=f"s3://sportsbar-fmcg-de-project-viswadithya/{data_source}/*.csv"

# print(base_path)

In [0]:
# df = (
#     spark.read.format("csv")
#         .option("header", True)
#         .option("inferSchema", True)
#         .load(base_path)
#         .withColumn("ingestion_time", F.current_timestamp())
#         .select("*", "_metadata.file_name", "_metadata.file_size")
# )

In [0]:
# df.printSchema()

In [0]:
# display(df.limit(10))

In [0]:
# df.write\
#  .format("delta") \
#  .option("delta.enableChangeDataFeed", "true") \
#  .mode("overwrite") \
#  .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
# df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")


# df_bronze.show(10)


###Data skew handling - uncomment the beloew lines, when you face data skew

In [0]:
# # Read sales data
# df = spark.read.csv(
#     "/FileStore/sales_data_skewed_product.csv",
#     header=True,
#     inferSchema=True
# )

# # Step 1: Identify the skewed product
# df.groupBy("product_id").count().orderBy(F.desc("count")).show()

# # Step 2: Add a salt key for the skewed product
# df_salted = df.withColumn(
#     "salt_key",
#     F.when(
#         F.col("product_id") == "P101",
#         F.floor(F.rand() * 10)
#     ).otherwise(F.lit(0))
# )

# # Step 3: Perform aggregation using product_id and salt_key
# df_partial = df_salted.groupBy(
#     "product_id", "salt_key"
# ).agg(
#     F.sum("sales_amount").alias("partial_sales")
# )

# # Step 4: Combine partial results
# df_final = df_partial.groupBy("product_id").agg(
#     F.sum("partial_sales").alias("total_sales")
# )

# df_final.show()

In [0]:
df_bronze=spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze=df_bronze.withColumn("ingestion_timestamp", F.current_timestamp())
display(df_bronze.limit(10))

### Invalid records

In [0]:
# df=df_bronze.groupBy("product_id").count().filter(F.col("count")>1)
# display(df)

In [0]:
df_invalid_products = df_bronze.filter(
    F.col("product_id").isNull()
    | F.col("product_name").isNull()
    | F.col("category").isNull()
)

In [0]:
df_invalid_products=df_invalid_products.withColumn(
    "rejection_reason",
    F.when(F.col("product_id").isNull(), "Missing product id")
    .when(F.col("product_name").isNull(), "Missing Product Name")
    .when(F.col("category").isNull(), "Missing category")
)

In [0]:
display(df_invalid_products)


## Quarantine

In [0]:
df_invalid_products.write\
    .format("delta")\
        .mode("append")\
            .option("mergeSchema", True)\
            .saveAsTable(f"{catalog}.{silver_schema}.quarantine_{data_source}"
)

In [0]:
df_valid_products = df_bronze.filter(
    F.col("product_id").isNotNull()
    & F.col("product_name").isNotNull()
    & F.col("category").isNotNull()
)

Cdc_timestamp=F.date_sub(F.current_timestamp(), 1)
df_valid_products=df_valid_products.filter(F.col("ingestion_timestamp")>Cdc_timestamp)


display(df_valid_products)

In [0]:
display(df_valid_products)

In [0]:
print("Total Bronze records:", df_bronze.count())
print("Valid records:", df_valid_products.count())
print("Invalid records:", df_invalid_products.count())

#Ingestion [ Capturing all records including late arriving records]

In [0]:
display(df_valid_products)


###removig duplicates

In [0]:
w=Window.partitionBy(F.col("product_id")).orderBy(F.col("updated_at").desc())

df_valid_products=df_valid_products.withColumn("rn",
                    F.row_number().over(w)
                                               )

display(df_valid_products)                                               

In [0]:
df_valid_products=df_valid_products.filter(F.col("rn")==1).drop("rn")
display(df_valid_products)

df_silver=df_valid_products

In [0]:
#deleted records

deleted_df=df_silver.filter(
    F.col("status")=='NON_ACTIVE'
)

In [0]:
# Fetching active records

df_silver=df_silver.filter(
    F.col("status")=='ACTIVE'
)


In [0]:
df_duplicate_products=df_valid_products.groupBy("product_id").count().filter(F.col("count")>1)

In [0]:
display(df_duplicate_products)

In [0]:
# print('Rows before duplicates dropped: ', df_valid_products.count())
# df_silver = df_valid_products.dropDuplicates(['product_id'])
# print('Rows after duplicates dropped: ', df_silver.count())

In [0]:
df_silver.select("category").distinct().show()

In [0]:
# Title case fix
df_silver = df_silver.withColumn(
    "category",
    F.when(F.col("category").isNull(), None)
     .otherwise(F.initcap("category"))
)

In [0]:
df_silver.select('category').distinct().show()

In [0]:
# Replace 'protien' → 'protein' in both product_name and category
df_silver = (
    df_silver
    .withColumn(
        "product_name",
        F.regexp_replace(F.col("product_name"), "(?i)Protien", "Protein")
    )
    .withColumn(
        "category",
        F.regexp_replace(F.col("category"), "(?i)Protien", "Protein")
    )
)


In [0]:
display(df_silver.limit(5))

In [0]:
### 1: Add division column
df_silver = (
    df_silver
    .withColumn(
        "division",
        F.when(F.col("category") == "Energy Bars",        "Nutrition Bars")
         .when(F.col("category") == "Protein Bars",       "Nutrition Bars")
         .when(F.col("category") == "Granola & Cereals",  "Breakfast Foods")
         .when(F.col("category") == "Recovery Dairy",     "Dairy & Recovery")
         .when(F.col("category") == "Healthy Snacks",     "Healthy Snacks")
         .when(F.col("category") == "Electrolyte Mix",    "Hydration & Electrolytes")
         .otherwise("Other")
    )
)


### 2: Variant column
df_silver = df_silver.withColumn(
    "variant",
    F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1)
)


### 3: Create new column: product_code  

# Invalid product_ids are replaced with a fallback value to avoid losing fact records and ensure downstream joins remain consistent

df_silver = (
    df_silver
    # 1. Generate deterministic product_code from product_name
    .withColumn(
        "product_code",
        F.sha2(F.col("product_name").cast("string"), 256)
    )
    # 2. Clean product_id: keep only numeric IDs, else set to 999999
    .withColumn(
        "product_id",
        F.when(
            F.col("product_id").cast("string").rlike("^[0-9]+$"),
            F.col("product_id").cast("string")
        ).otherwise(F.lit(999999).cast("string"))
    )
    # 3. Rename product_name → product
    .withColumnRenamed("product_name", "product")
)

In [0]:
display(df_silver.limit(5))

In [0]:
df_silver = df_silver.select("product_code", "division", "category", "product", "variant", "product_id", "ingestion_timestamp")

In [0]:
# df_silver.write\
#  .format("delta") \
#  .option("delta.enableChangeDataFeed", "true") \
#  .option("mergeSchema", "true") \
#  .mode("overwrite") \
#  .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

In [0]:
display(df_silver.limit(5))

In [0]:
silver_table = f"{catalog}.{silver_schema}.{data_source}"

# spark.sql(f"""
#     ALTER TABLE {silver_table}
#     ADD COLUMNS (
#         effective_start_date TIMESTAMP,
#         effective_end_date TIMESTAMP,
#         is_current BOOLEAN
#     )
# """)

In [0]:
silver_delta = DeltaTable.forName(
    spark,
    silver_table
)

current_active_df = (
    spark.table(silver_table)
    .filter(
        (F.col("is_current") == True)
    )
    .cache()
)

In [0]:
deletions_to_process_df = (
    deleted_df.alias("b")
    .join(
        current_active_df.alias("s"),
        F.col("b.product_id") == F.col("s.product_id"),
        "inner"
    )
    .select(
        F.col("b.product_id"),
        F.col("b.updated_at").alias("delete_timestamp")
    )
    .dropDuplicates(["product_id"])
)

In [0]:
current_active_df.count()

In [0]:
silver_delta = DeltaTable.forName(spark, silver_table)

(
    silver_delta.alias("s")
    .merge(
        deletions_to_process_df.alias("d"),
        """
        s.product_id = d.product_id
        AND s.is_current = true
        """
    )
    .whenMatchedUpdate(
        condition="d.delete_timestamp > s.ingestion_timestamp",
        set={
            "is_current": "false",
            "effective_end_date": "d.delete_timestamp"
        }
    )
    .execute()
)

In [0]:
silver_table = f"{catalog}.{silver_schema}.{data_source}"

# 1. Read the incoming, already-validated DataFrame
bronze_df = df_silver

# 2. Keep the latest event for each product
window_spec = (
    Window
    .partitionBy("product_id")
    .orderBy(F.col("ingestion_timestamp ").desc_nulls_last())
)

bronze_df = (
    bronze_df
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

# 3. Separate active and deleted products
active_df = bronze_df.filter(
    F.upper(F.trim(F.col("status"))) == "ACTIVE"
)

deleted_df = bronze_df.filter(
    F.upper(F.trim(F.col("status"))) == "NON_ACTIVE"
)

# 4. Read current Silver records
silver_df = spark.table(silver_table).filter(
    F.col("is_current") == True
)

# 5. Find changed active products
joined_df = active_df.alias("b").join(
    silver_df.alias("s"),
    F.col("b.product_id") == F.col("s.product_id"),
    "inner"
)

changed_df = joined_df.filter(
    ~(
        F.col("b.product").eqNullSafe(F.col("s.product"))
        & F.col("b.category").eqNullSafe(F.col("s.category"))
    )
).select("b.*")

# 6. Find new active products
new_df = active_df.join(
    silver_df.select("product_id").distinct(),
    on="product_id",
    how="left_anti"
)

# 7. Prepare new SCD Type 2 versions
def prepare_new_version(df):
    return (
        df
        .withColumn(
            "effective_start_date",
            F.col("ingestion_timestamp").cast("timestamp")
        )
        .withColumn(
            "effective_end_date",
            F.lit(None).cast("timestamp")
        )
        .withColumn("is_current", F.lit(True))
    )

changed_versions = prepare_new_version(changed_df)
new_versions = prepare_new_version(new_df)

# 8. Stage records for the existing SCD Type 2 MERGE
expire_rows = changed_versions.withColumn(
    "merge_key", F.col("product_id")
)

insert_changed_rows = changed_versions.withColumn(
    "merge_key",
    F.lit(None).cast(changed_versions.schema["product_id"].dataType)
)

insert_new_rows = new_versions.withColumn(
    "merge_key",
    F.lit(None).cast(new_versions.schema["product_id"].dataType)
)

staged_df = (
    expire_rows
    .unionByName(insert_changed_rows)
    .unionByName(insert_new_rows)
)

# 9. Existing SCD Type 2 MERGE
delta_silver = DeltaTable.forName(spark, silver_table)

(
    delta_silver.alias("t")
    .merge(
        staged_df.alias("s"),
        """
        t.product_id = s.merge_key
        AND t.is_current = true
        """
    )
    .whenMatchedUpdate(
        condition="s.merge_key IS NOT NULL",
        set={
            "effective_end_date": "s.effective_start_date",
            "is_current": "false"
        }
    )
    .whenNotMatchedInsert(
        values={
            "product_code": "s.product_code",
            "division": "s.division",
            "category": "s.category",
            "product": "s.product",
            "variant": "s.variant",
            "product_id": "s.product_id",
            "ingestion_timestamp": "s.ingestion_timestamp",
            "effective_start_date": "s.effective_start_date",
            "effective_end_date": "s.effective_end_date",
            "is_current": "s.is_current"
        }
    )
    .execute()
)



In [0]:
# # 10. Handle deleted products separately
# deletions_to_process_df = (
#     deleted_df
#     .select(
#         "product_id",
#         F.col("ingestion_timestamp").alias("delete_timestamp")
#     )
#     .join(
#         spark.table(silver_table)
#         .filter(F.col("is_current") == True)
#         .select("product_id", "ingestion_timestamp"),
#         on="product_id",
#         how="inner"
#     )
#     .filter(
#         F.col("delete_timestamp") > F.col("ingestion_timestamp")
#     )
#     .select("product_id", "delete_timestamp")
#     .dropDuplicates(["product_id"])
# )

# delta_silver = DeltaTable.forName(spark, silver_table)

# (
#     delta_silver.alias("t")
#     .merge(
#         deletions_to_process_df.alias("d"),
#         """
#         t.product_id = d.product_id
#         AND t.is_current = true
#         """
#     )
#     .whenMatchedUpdate(
#         condition="d.delete_timestamp > t.ingestion_timestamp",
#         set={
#             "effective_end_date": "d.delete_timestamp",
#             "is_current": "false"
#         }
#     )
#     .execute()
# )

In [0]:
deleted_df.write\
    .mode("append")\
        .option("mergeSchema", "true")\
            .saveAsTable("{catalog}.{silver_schema}.Tombstone{data_source}")

In [0]:
# # silver_table=f"{catalog}.{silver_schema}.{data_source}"

# # spark.sql(f"""
# #     UPDATE {silver_table}
# #     SET
# #         effective_start_date = ingestion_timestamp,
# #         effective_end_date = NULL,
# #         is_current = true
# # """)
# bronze_df = df_silver



# silver_df=(
#     spark.table(silver_table).filter(F.col("is_current")==True)
# )
# joined_df=bronze_df.alias("b").join(
#     silver_df.alias("s"),
#     F.col("b.product_id") == F.col("s.product_id"),
#     "inner"
# )

# changed_df = joined_df.filter(
#     ~(
#         F.col("b.product").eqNullSafe(F.col("s.product"))
#         & F.col("b.category").eqNullSafe(F.col("s.category")) 
#     )
# ).select("b.*")

# new_df = (
#     bronze_df.join(
#         silver_df.select("product_id"),
#         on="product_id",
#         how="left_anti"
#     )
# )

# def prepare_new_version(df):
#     return (
#         df
#         .withColumn(
#             "effective_start_date",
#             F.coalesce(
#                 F.col("ingestion_timestamp").cast("timestamp"),
#                 F.current_timestamp()
#             )
#         )
#         .withColumn(
#             "effective_end_date",
#             F.lit(None).cast("timestamp")
#         )
#         .withColumn("is_current", F.lit(True))
#     )

# changed_versions = prepare_new_version(changed_df)
# new_versions = prepare_new_version(new_df)

# expire_rows = (
#     changed_versions
#     .withColumn("merge_key", F.col("product_id"))
# )

# insert_changed_rows = (
#     changed_versions
#     .withColumn("merge_key", F.lit(None).cast("string"))
# )

# insert_new_rows = (
#     new_versions
#     .withColumn("merge_key", F.lit(None).cast("string"))
# )

# staged_df = (
#     expire_rows
#     .unionByName(insert_changed_rows)
#     .unionByName(insert_new_rows)
# )



In [0]:
# delta_silver = DeltaTable.forName(spark, silver_table)

# (
#     delta_silver.alias("t")
#     .merge(
#         staged_df.alias("s"),
#         """
#         t.product_id == s.merge_key
#         AND t.is_current = true
#         """
#     )
#     .whenMatchedUpdate(
#         condition="s.merge_key IS NOT NULL",
#         set={
#             "effective_end_date": "s.effective_start_date",
#             "is_current": "false"
#         }
#     )
#     .whenNotMatchedInsert(
#         values={
#             "product_code": "s.product_code",
#             "division": "s.division",
#             "category": "s.category",
#             "product": "s.product",
#             "variant": "s.variant",
#             "product_id": "s.product_id",
#             "ingestion_timestamp": "s.ingestion_timestamp",
#             "effective_start_date": "s.effective_start_date",
#             "effective_end_date": "s.effective_end_date",
#             "is_current": "s.is_current"
#         }
#     )
#     .execute()
# )

In [0]:
# target_table=f"{catalog}.{silver_schema}.{data_source}"

# if not spark.catalog.tableExists(target_table):
#     (
#         df_silver.write\
#             .format("delta")\
#                 .mode("overwrite")\
#                     .option("delta.enableChangeDataFeed", True)\
#                         .option("overWriteSchema", True)\
#                             .saveAsTable(target_table)

#     )

# else:
#     delta_table=DeltaTable.forName(spark, target_table)
#     (
#         delta_table.alias("target").merge(
#             df_silver.alias("source"),
#             "target.product_id=source.product_id"
#         ).whenMatchedUpdateAll(
#             condition="""
#                 source.updated_at > target.updated_at
#             """
#         )
#     ).whenNotMatchedInsertAll()\
#         .execute


In [0]:
# target_table = f"{catalog}.gold.dim_products"

In [0]:
%skip
df_target = spark.table(target_table)

In [0]:
# df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")
# df_gold = df_silver.select("product_code", "product_id", "division", "category", "product", "variant","product_status")
# df_gold.show(5)

In [0]:
# df_gold=df_gold.withColumn(
#     "product_status",
#     f.upper(f.trim(f.col("product_status")))
# )

In [0]:
# invalid_status=df_gold.filter(
#     ~f.col("product_status").isin(""ACTIVE", "DISCONTINUED"")|
#     f.col("product_status").isNull()
# )

# if invalid_status.limit(1).count()>0:
#     raise ValueError

# df_gold=(
#     df_gold.withColumn(
#         "is_active",
#         f.col("product_status")=="Active"
#     )
# ).withColumn(
#     "deactivated_at",
#     f.when(
#         f.col("product_status")="DISCONTINUED",
#         f.current_timestamp()
#     ).otherwise(f.lit(None).cast(timestamp()))
# )

In [0]:
# df_compare = (
#     df_gold.alias("s")
#     .join(
#         df_target.alias("t"),
#         on="product_code",
#         how="left"
#     )
# )

In [0]:
# df_changes = df_compare.filter(
#     f.col("t.product_code").isNull()
#     | ~(
#         f.col("s.product_id").eqNullSafe(f.col("t.product_id"))
#         & f.col("s.division").eqNullSafe(f.col("t.division"))
#         & f.col("s.category").eqNullSafe(f.col("t.category"))
#         & f.col("s.product").eqNullSafe(f.col("t.product"))
#         & f.col("s.variant").eqNullSafe(f.col("t.variant"))
#     )
# ).select("s.*")

# df_changes.show()

In [0]:
# df_gold.write\
#  .format("delta") \
#  .option("delta.enableChangeDataFeed", "true") \
#  .mode("overwrite") \
#  .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
# delta_table = DeltaTable.forName(spark, "fmcg_de_project.gold.dim_products")
# # df_child_products = spark.sql(f"SELECT product_code, division, category, product, variant FROM fmcg_de_project.gold.sb_dim_products;")

# #SCD TYPE-2

# child_table="fmcg_de_project.gold.sb_dim_products"
# existing_cols=spark.table(child_table).columns

# if "effective_start_date" not in existing_cols:
#     spark.sql(f"""
#               alter table {child_table}
#               add columns (
#                   effective_start_date timestamp,
#                   effective_end_date timestamp,
#                   is_current boolean
#               )
#               """)
    
#     spark.sql(f"""
#               update {child_table}
#               set effective_start_date=current_timestamp(),
#               effective_end_date = TIMESTAMP '9999-12-31 00:00:00',
#               is_current=True
#             where   
#                 effective_start_date is Null
#               """)
    
# df_child_products = spark.table(child_table)

# df_child_products.show(5)

In [0]:
# df_latest = df_child.filter(
#     f.col("is_current") == "True"
# )

In [0]:
%skip
df_child_products.count()

In [0]:
# delta_table.alias("target").merge(
#     source=df_latest.alias("source"),
#     condition="target.product_code = source.product_code"
# ).whenMatchedUpdate(
#     set={
#         "division": "source.division",
#         "category": "source.category",
#         "product": "source.product",
#         "variant": "source.variant"
#     }
# ).whenNotMatchedInsert(
#     values={
#         "product_code": "source.product_code",
#         "division": "source.division",
#         "category": "source.category",
#         "product": "source.product",
#         "variant": "source.variant"
#     }
# ).execute()